# 05 — Validate the from-scratch result and create a compact output bundle

The reference JSON is used **only after** the calculation, to identify accidental pipeline drift. It is never read by the simulation, analysis, fitting, or figure notebooks. Platform-dependent floating-point differences are expected, so the comparison is diagnostic rather than bitwise.

In [1]:
from pathlib import Path
import os, sys

def find_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Repository root not found. Start Jupyter inside the extracted directory.")

ROOT = find_root(Path.cwd())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("ROOT =", ROOT)


ROOT = /mnt/data/work_release_v3/LagrangianEllipsoid_Corrected_Release_v3/2_github_repository


In [2]:
import json, math
new=json.loads((ROOT/"data"/"results"/"paper_numbers.json").read_text())
ref=json.loads((ROOT/"reference"/"paper_numbers_v2.0.0.json").read_text())

paths=[
 ("saturation.slope_3_8_MEE", new["saturation"]["slope_3_8_MEE"], ref["saturation"]["slope_3_8_MEE"]),
 ("saturation.slope_3_8_mass", new["saturation"]["slope_3_8_mass"], ref["saturation"]["slope_3_8_mass"]),
 ("saturation.mean_sigma_MEE", new["saturation"]["mean_sigma_MEE"], ref["saturation"]["mean_sigma_MEE"]),
 ("saturation.mean_sigma_mass", new["saturation"]["mean_sigma_mass"], ref["saturation"]["mean_sigma_mass"]),
 ("saturation.delta_rho", new["saturation"]["delta_rho"], ref["saturation"]["delta_rho"]),
]
rows=[]
for name,a,b in paths:
    rows.append((name,a,b,a-b,abs(a-b)/(abs(b)+1e-12)))
import pandas as pd
df=pd.DataFrame(rows,columns=["quantity","new","reference","difference","relative_difference"])
display(df)
# Re-running the exact seeds and parameters should normally be much closer than this broad guardrail.
assert (df.relative_difference < 0.10).all(), "A key result differs by more than 10%; inspect environment and logs."
print("Key numerical comparison passed.")

,quantity,new,reference,difference,relative_difference
0,saturation.slope_3_8_MEE,0.028068,0.028068,0.0,0.0
1,saturation.slope_3_8_mass,0.069822,0.069822,0.0,0.0
2,saturation.mean_sigma_MEE,0.909892,0.909892,0.0,0.0
3,saturation.mean_sigma_mass,1.069604,1.069604,0.0,0.0
4,saturation.delta_rho,0.337982,0.337982,0.0,0.0


Key numerical comparison passed.


In [3]:
import subprocess
proc=subprocess.run([sys.executable,"-m","pytest","-q"],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError("Release checks failed")

...........                                                              [100%]
11 passed in 0.96s



In [4]:
import zipfile, hashlib, json
from datetime import datetime, timezone

export=ROOT/"from_scratch_output_bundle.zip"
include_roots=[ROOT/"configs",ROOT/"data"/"results",ROOT/"figures",ROOT/"tables",ROOT/"logs"]
with zipfile.ZipFile(export,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for base in include_roots:
        for p in sorted(base.rglob("*")):
            if p.is_file() and p.name != ".gitkeep":
                z.write(p,p.relative_to(ROOT))
sha=hashlib.sha256(export.read_bytes()).hexdigest()
(ROOT/"from_scratch_output_bundle.sha256").write_text(sha+"  "+export.name+"\n")
print("Created",export)
print("SHA-256",sha)
print("Raw trajectories are intentionally not duplicated in this compact export; they remain in data/raw/.")

Created /mnt/data/work_release_v3/LagrangianEllipsoid_Corrected_Release_v3/2_github_repository/from_scratch_output_bundle.zip
SHA-256 c8d91e8603770d4892f904d81e04272ffc995c8159ddd2f811093ed3c6fedaaa
Raw trajectories are intentionally not duplicated in this compact export; they remain in data/raw/.
